# 1010 — SDS-PAGE annotation: e281 Ni-NTA purification fractions

Annotates the 2026-10-10 Coomassie gel of the `CDS12_ANC_Node281` (`e281`) Ni-NTA purification and
asks whether anything came off the column. Same pipeline as
[`0929_gel_annotation.ipynb`](0929_gel_annotation.ipynb); the three differences are listed in §2.

## Verdict: nothing was purified, and nothing bound

| | |
|---|---|
| **Elutions E1, E2** | **Nothing anywhere on either lane.** Strongest 30–100 kDa peak has prominence 0.011 against a noise floor of 0.020 measured on this gel's own five unloaded lanes. Lane totals 13.4 and 12.9 sit *inside* the unloaded-lane range (13.9–22.4). The elution lanes are not distinguishable from blank gel. |
| **Load (`raw`) vs flow-through (`FT1`)** | **The same band at the same strength.** 40–44 kDa, prominence 0.040 and 0.036, band volume 1.09 and 1.01 — an 8 % difference. Lane totals 24.3 and 21.3. Nothing measurable was taken out of the load by the column. |
| **Wash W3** | Trace at 42–46 kDa, prominence 0.019, just under the floor. That is what a third wash should look like. |
| **The 40–44 kDa band itself** | It lands on the predicted 42.27 kDa — but it is in the flow-through just as strongly, so it did not bind the resin. Every construct carries an N-terminal His6, so the construct is supposed to bind. This band behaves like a host protein that happens to run near 42 kDa. |

This agrees with the initial read at the bench: no purified protein.

## What this gel can and cannot say

**It can say** that nothing eluted and that nothing was retained from the load. That conclusion does
not depend on which of the causes below is the real one.

**It cannot separate the three causes**, and nothing on this gel narrows them down:

1. **No e281 protein was made.** The 0929 crude gel already found no band at 42.3 kDa in e281
   lysate, induced or not ([`0929_gel_annotation.ipynb`](0929_gel_annotation.ipynb)). The ANC
   inserts are `Codon Optimized: No`, so rare-codon stalling is the standing first suspect.
2. **It was made but is insoluble.** No pellet / insoluble lane was run, so an inclusion-body
   product is invisible here.
3. **It was made and soluble, and the column failed.** No construct with known binding was run over
   the resin alongside, so there is no positive control for the purification itself.

**The load lane is faint.** Its total is 24.3 against 48.7 for the ladder lane. A product at a few
per cent of an already thin load would sit under the floor in *every* lane, `raw` included.

## What to run next

1. **Show the protein exists before purifying it again — UNDONE.** Crude induced/uninduced pair plus
   the blank BL21(DE3) ±IPTG lanes (the 10/08 gel plan in
   [`2026-10-05_last_2_week_lab_planning.md`](../documents/2026-10-05_last_2_week_lab_planning.md),
   still not run), or an anti-His western on crude lysate.
2. **Keep and run the insoluble pellet.** One extra lane decides cause 2.
3. **Run W1 and W2 as well.** Only W3 was loaded, so it is not known whether anything came off in an
   earlier wash.
4. **Load more.** Half the ladder's total is not enough to call a negative on a minor species.
5. **Purify CDS12_WT first, not e281.** It is the only construct with an induction-dependent band
   (0929), and the planning doc already puts it in batch 1 with E1; e281 is batch 2. Purifying the
   one construct with no evidence of expression tests the column and the protein at the same time
   and can only give an uninterpretable answer.

## Inputs

| File | Original | What |
|---|---|---|
| `data/assays/1010_sds_page/nta-fractions.jpeg` | `1010_sdsGel/IMG_9595.jpeg` | the gel |
| `data/assays/1010_sds_page/nta-fractions_label.jpeg` | `1010_sdsGel/IMG_9592.jpeg` | handwritten loading map |

Originals are left in place; the copies are renamed, not edited. The gel was photographed
**2026-10-11 01:24 local** (EXIF), so it was run on the evening of 10/10. Predicted mass comes from
[`0910_construct_protein_mw.ipynb`](0910_construct_protein_mw.ipynb): **42.27 kDa**, including
pET-28a's 21 aa `MGSSHHHHHHSSGLVPRGSHM` leader.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageOps
from matplotlib.patches import Polygon
from scipy.ndimage import gaussian_filter, gaussian_filter1d, percentile_filter
from scipy.signal import find_peaks

ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
IMG  = ROOT/"data/assays/1010_sds_page"
OUT  = ROOT/"code/output/1010_gel_annotation"; OUT.mkdir(parents=True, exist_ok=True)

FN        = "nta-fractions.jpeg"
PREDICTED = 42.27        # ANC_Node281 incl. the pET-28a His6 leader (0910 notebook)

_img = None
def load():
    "Phone photo carries an EXIF rotation; without transpose the gel comes out landscape."
    global _img
    if _img is None:
        _img = np.asarray(ImageOps.exif_transpose(Image.open(IMG/FN)).convert("RGB")).astype(float)/255.
    return _img

print(load().shape)

(4032, 3024, 3)


## 1. Loading plan, transcribed from the label sheet

Handwritten, on the same page as a Bradford/OD@595 series.

| Lane | Sample |
|---|---|
| 1 | `ladder` — **written, then struck out** |
| 2 | — |
| 3 | Ladder |
| 4 | `raw` |
| 5 | `FT1` — flow-through |
| 6 | `W3` — wash 3 |
| 7 | `E1` — elution 1 |
| 8 | `E2` — elution 2 |
| 9 | Ladder |
| 10–12 | — |

> ⚠️ **Ambiguities, surfaced before anything is measured.**
> 1. **The construct is not written on the sheet.** Identified as **e281** by the user, not by the
>    sheet. Nothing in the photograph records it.
> 2. **No date, gel %, voltage/time, sample volume, or buffer composition.** The imidazole
>    concentration in the wash and in the elution buffer is not recorded anywhere, so a failed
>    elution cannot be told from an over-stringent wash.
> 3. **"raw" is not defined** — whole-cell lysate or cleared lysate. If cleared, an insoluble
>    product never reached the column and "nothing bound" would be the expected result.
> 4. **W1, W2 and E3 were not loaded.** Only one wash and two elutions are on the gel.
> 5. **Name clash.** `E1`/`E2` here are *elution fractions*. `e1`/`e40`/`e281` elsewhere in this repo
>    are *ancestral nodes*. The planning doc flags this same clash.
> 6. **Five of twelve wells were left unloaded.** The missing control lanes — pellet, uninduced,
>    blank BL21(DE3) — would have cost nothing to add.
> 7. The sheet's OD@595 series (`raw` 2.068, `1:2` 1.797, `1:4` 1.355, `1:8` 1.110) is **not used
>    here**: no standard curve was recorded, and the readings do not halve with dilution, so the
>    neat reading is likely past the linear range. Transcribed for the record only.

**Orientation is checked, not assumed** (§5). The two ladders are 2 lanes in from the left and
3 lanes in from the right. A left–right flip would put them at lanes 4 and 10, which is not what the
sheet says, so the asymmetry fixes the orientation.

## 2. Background correction — what changed from 0929

Same method as 0929: the background is the **80th percentile in a 400 px window**, then smoothed,
and signal is optical density, `OD = −log10(I / background)`. A window wider than a lane always
contains clear gel, so the percentile picks the gel rather than the sample.

Three things differ from 0929, all forced by the photograph and the experiment:

1. **The noise floor is measured, not assumed.** 0929 had to guess a floor of 0.055 from the paper
   texture under the tray. This gel has **five genuinely unloaded lanes**, so the floor is read
   straight off them (§6). It comes out at 0.020 — the tray here sits on a smooth, evenly lit
   surface. Detection on this gel is therefore more sensitive than on 0929, which makes the empty
   elution lanes a stronger negative.
2. **Lanes 3 and 9 are the ladders**, not 1 and 12, and five lanes are blank. The lane grid is built
   from the two ladder positions and extended outwards.
3. **One construct, one block.** No second construct to compare against on the same gel.

**OD totals are not numerically comparable with the 0915 or 0929 tables** — different illumination,
different exposure, different background estimate.

Every figure is produced twice, as before:

- `_uniform` — a single linear contrast stretch, nothing else. Publication-safe under iGEM's image
  rules; use this one on the wiki.
- `_flattened` — background-divided, for visibility, disclosed in the caption.

No local, per-lane or per-band edits anywhere.

In [2]:
# gel body crop: below the wells, above the dye front
BOX = (80, 2960, 1080, 3320)          # x0, x1, y0, y1

def crop():
    x0, x1, y0, y1 = BOX
    return load()[y0:y1, x0:x1]

def background(ch, win=400, pct=80, ds=8):
    "Clear-gel level: 80th percentile in a window wider than a lane, computed on a /8 thumbnail."
    small = gaussian_filter(ch, ds)[::ds, ::ds]
    bg    = gaussian_filter(percentile_filter(small, pct, size=win//ds), win/ds/4)
    im    = Image.fromarray(bg.astype(np.float32)).resize(ch.shape[::-1], Image.BILINEAR)
    return np.clip(np.asarray(im), 1e-3, None)

_od = None
def od():
    global _od
    if _od is None:
        lum = crop().mean(2)
        _od = -np.log10(np.clip(lum/background(lum), 1e-3, None))   # ~0 = clear gel, >0 = band
    return _od

H, W = od().shape
print(f"gel body crop: {W} x {H} px")

gel body crop: 2880 x 2240 px


## 3. Which ladder is this?

Same test as 0915 and 0929, adapted to a stained gel: the whole gel is blue, so absolute colour is
useless, and each band's red and green excess is taken **relative to the median band in its own
lane**.

In [3]:
ROUGH_X = dict(L=668, R=2005)

def ladder_bands(w=45):
    g, o = crop(), od(); rows = []
    for tag, lx in ROUGH_X.items():
        prof  = gaussian_filter1d(o[:, lx-w:lx+w].mean(1), 6)
        bp, _ = find_peaks(prof, prominence=0.015, distance=40)
        rgb   = np.array([g[p-8:p+8, lx-w:lx+w].reshape(-1, 3).mean(0) for p in bp])
        red   = rgb[:,0] - (rgb[:,1] + rgb[:,2])/2
        green = rgb[:,1] - (rgb[:,0] + rgb[:,2])/2
        red, green = red - np.median(red), green - np.median(green)
        for i, p in enumerate(bp):
            rows.append(dict(lane=tag, band=i+1, y=int(p), od=round(float(prof[p]), 3),
                             colour="RED" if red[i] > .10 else ("GREEN" if green[i] > .08 else "blue"),
                             red_excess=round(float(red[i]), 3), green_excess=round(float(green[i]), 3)))
    return pd.DataFrame(rows)

bands = ladder_bands()
print(bands.groupby("lane").size().to_string(), "\n")
print(bands[bands.colour != "blue"].to_string(index=False))

lane
L    10
R    10 

lane  band    y    od colour  red_excess  green_excess
   L     4 1015 0.133    RED       0.200        -0.016
   L    10 2207 0.156  GREEN       0.031         0.160
   R     4  987 0.142    RED       0.195        -0.003
   R    10 2200 0.147  GREEN       0.030         0.146


Both ladder lanes: **10 bands, the 4th from the top red, the lowest green.** Same **Thermo PageRuler
Prestained Protein Ladder** as 0915 and 0929 — 170, 130, 100, **70 (orange)**, 55, 40, 35, 25, 15,
**10 (green)** kDa.

In [4]:
LADDER_MW = np.array([170,130,100,70,55,40,35,25,15,10], float)
LOGMW     = np.log10(LADDER_MW)

# Band y-positions as detected above, for the two ladder lanes (3 and 9).
CAL = dict(yL=[507,597,810,1015,1272,1522,1735,1887,2108,2207],
           yR=[491,582,789,987,1245,1496,1712,1866,2102,2200])

for tag, key in (("L","yL"), ("R","yR")):          # hard-coded values must be what was detected
    found = bands[bands.lane == tag].y.tolist()
    assert found == CAL[key], (tag, found)
print("CAL matches the detected bands on both ladder lanes")

CAL matches the detected bands on both ladder lanes


## 4. Calibration and its precision

Same two calibrations as 0929, kept side by side because they disagree where it matters:

- **cubic** — `log10(MW)` against migration, cubic fit. The 0915 method, kept for comparability.
- **rung-to-rung** — log-linear interpolation between the two flanking rungs. Exact at the rungs.

Each rung is interpolated between its own left and right position rather than using one global tilt.

In [5]:
rows = []
for tag, key in (("left","yL"), ("right","yR")):
    y = np.array(CAL[key], float); c3 = np.polyfit(y, LOGMW, 3)
    fit = 10**np.polyval(c3, y)
    rows.append(dict(lane=f"ladder {tag}", max_err=f"{np.abs(fit/LADDER_MW-1).max()*100:.0f}%",
                     **{f"{m:.0f}": round(f/m, 2) for m, f in zip(LADDER_MW, fit)}))
pd.set_option("display.width", 250)
print("cubic fit / nominal mass, per rung")
print(pd.DataFrame(rows).to_string(index=False))
print("\nSame residual pattern as 0915 and 0929 -> a property of this ladder, not noise."
      "\nWorking precision stays +/-9%, i.e. +/-4 kDa on a 42 kDa target.")

cubic fit / nominal mass, per rung
        lane max_err  170  130  100   70   55   40   35   25   15   10
 ladder left      9% 0.99 1.05 0.92 1.01 1.01 1.09 0.94 1.00 0.96 1.04
ladder right      8% 1.00 1.05 0.93 1.02 1.00 1.08 0.94 1.01 0.95 1.05

Same residual pattern as 0915 and 0929 -> a property of this ladder, not noise.
Working precision stays +/-9%, i.e. +/-4 kDa on a 42 kDa target.


## 5. Lane grid, and the orientation check

The comb is 12 wells. The ladders are **lanes 3 and 9**, so the grid is built from those two
positions and extended outwards — the reverse of 0929, where the ladders were the outermost lanes.

Two independent checks that this assignment is right, neither of which uses the label sheet:

- **The grid sits centred in the gel.** The gel's own left and right edges are found from the
  luminance step at the gel boundary. A 12-lane comb centred between them should leave equal margins.
- **The asymmetry fixes left from right.** The ladders are 2 lanes in from one edge and 3 from the
  other. Under a left–right flip they would read as lanes 4 and 10, which contradicts the sheet.

In [6]:
NL          = 12
LADDER_IDX  = (2, 8)                                   # 0-based: lanes 3 and 9
LANE_X      = dict(top=(674, 1990), bottom=(654, 2031))   # ladder centre lines, fitted below

def lane_x(i, y):
    (l0, r0), (l1, r1) = LANE_X["top"], LANE_X["bottom"]
    xl, xr = l0 + (l1-l0)*y/H, r0 + (r1-r0)*y/H
    return xl + (xr-xl)*(i - LADDER_IDX[0])/(LADDER_IDX[1] - LADDER_IDX[0])

def pitch(y):   return (lane_x(LADDER_IDX[1], y) - lane_x(LADDER_IDX[0], y))/(LADDER_IDX[1] - LADDER_IDX[0])
def frac(i):    return (i - LADDER_IDX[0])/(LADDER_IDX[1] - LADDER_IDX[0])

def band_centre(y, x, half):
    "Horizontal centre of a band at row y: midpoint of its extent at half height."
    row  = gaussian_filter1d(od()[y-12:y+12].mean(0), 6)[int(x-half):int(x+half)]
    base = np.percentile(row, 10); idx = np.where(row - base > 0.5*(row.max() - base))[0]
    return (idx.min() + idx.max())/2 + int(x-half), idx.max() - idx.min()

for tag, key, rough in (("L","yL",668), ("R","yR",2005)):
    pts = np.array([(y, *band_centre(y, rough, 170)) for y in CAL[key]], float)
    ok  = np.abs(pts[:,2]/np.median(pts[:,2]) - 1) < 0.12      # drop bands merged with a neighbour
    c   = np.polyfit(pts[ok,0], pts[ok,1], 1)
    print(f"ladder {tag}: x(top) {np.polyval(c,0):.0f}  x(bottom) {np.polyval(c,H):.0f}"
          f"   from {ok.sum()} clean rungs")
print(f"lane pitch {pitch(H/2):.0f} px")

# gel edges, from the luminance step at the boundary, in crop coordinates
lum = load().mean(2); x0, y0 = BOX[0], BOX[2]
edges = []
for y in (1200, 2200, 3200):
    row = gaussian_filter1d(lum[y-20:y+20].mean(0), 15); d = np.diff(row)
    edges.append((int(np.argmin(d[:600])) - x0, int(np.argmax(d[2400:])) + 2400 - x0))
gl, gr = np.mean([e[0] for e in edges]), np.mean([e[1] for e in edges])
print(f"\ngel edges (crop x): {gl:.0f} .. {gr:.0f}   centre {np.mean([gl,gr]):.0f}")
print(f"12-lane grid:       {lane_x(0,H/2):.0f} .. {lane_x(NL-1,H/2):.0f}   centre "
      f"{(lane_x(0,H/2)+lane_x(NL-1,H/2))/2:.0f}")
print(f"margins: left {lane_x(0,H/2)-gl:.0f} px, right {gr-lane_x(NL-1,H/2):.0f} px, "
      f"against a {pitch(H/2):.0f} px pitch")
print("-> a 12-well comb centred in the gel, with the ladders at lanes 3 and 9 as the sheet says.")

ladder L: x(top) 676  x(bottom) 651   from 7 clean rungs
ladder R: x(top) 1989  x(bottom) 2037   from 6 clean rungs
lane pitch 224 px

gel edges (crop x): -55 .. 2924   centre 1434
12-lane grid:       215 .. 2684   centre 1449
margins: left 270 px, right 240 px, against a 224 px pitch
-> a 12-well comb centred in the gel, with the ladders at lanes 3 and 9 as the sheet says.


The grid centre lands within ~15 px of the gel centre and the two outer margins agree to about
0.13 of a lane pitch. The comb is 12 wells, centred, and the ladders are lanes 3 and 9.

## 6. Quantification

Per lane, in optical density, same definitions as 0929:

- **total** — integrated OD over the lane. Total protein loaded, nothing more.
- **contrast at the predicted mass** — peak height inside a ±8 % window around 42.27 kDa, above the
  median of the flanking gel.
- **strongest band, 30–100 kDa** — its position, its apparent mass by both calibrations, its
  prominence, and its volume (OD summed over ±45 rows, flanks subtracted).

**The noise floor is measured on this gel.** Lanes 1, 2, 10, 11 and 12 were never loaded, so their
strongest "bands" are pure noise. The floor is set just above the worst of them.

In [7]:
LANES = ["(empty)", "(empty)", "P.L", "raw", "FT1", "W3", "E1", "E2", "P.L",
         "(empty)", "(empty)", "(empty)"]
EMPTY   = [0, 1, 9, 10, 11]          # 0-based: the five unloaded lanes
SAMPLES = [3, 4, 5, 6, 7]            # 0-based: raw, FT1, W3, E1, E2
SEARCH  = (30, 100)

def rungs(f):
    yL, yR = np.array(CAL["yL"], float), np.array(CAL["yR"], float)
    return yL + f*(yR - yL)

def y_of(mw, f):        return float(np.polyval(np.polyfit(LOGMW, rungs(f), 3), np.log10(mw)))
def mw_cubic(y, f):     return float(10**np.polyval(np.polyfit(rungs(f), LOGMW, 3), y))
def mw_piecewise(y, f): return float(10**np.interp(y, rungs(f), LOGMW))

def profile(i, width=.275, shift=0.):
    "Mean OD across the lane, row by row, following the lane's centre line."
    o = od()
    col = np.array([o[y, int(lane_x(i,y) + pitch(y)*(shift-width)):
                         int(lane_x(i,y) + pitch(y)*(shift+width))].mean()
                    for y in range(H)])
    return gaussian_filter1d(col, 5)

def volume(i, p, hw=45, gap=20, fl=70):
    o = od(); x = lane_x(i, p); w = pitch(p)*.45
    sl = o[:, int(x-w):int(x+w)].mean(1)
    base = np.median(np.r_[sl[p-hw-gap-fl:p-hw-gap], sl[p+hw+gap:p+hw+gap+fl]])
    return float((sl[p-hw:p+hw] - base).sum())

def strongest(col, f):
    lo, hi = y_of(SEARCH[1], f), y_of(SEARCH[0], f)
    pk, pr = find_peaks(col, prominence=0.004, distance=15)
    return max([(p, q) for p, q in zip(pk, pr["prominences"]) if lo < p < hi], key=lambda t: t[1])

# --- noise floor, from the five unloaded lanes -------------------------------
zone = slice(int(.02*H), int(.97*H))
noise = [strongest(profile(i), frac(i))[1] for i in EMPTY]
FLOOR = round(max(noise) + 0.005, 3)
print("strongest 30-100 kDa peak in each UNLOADED lane: " +
      ", ".join(f"lane {i+1} {q:.3f}" for i, q in zip(EMPTY, noise)))
print(f"-> noise floor {FLOOR}  (worst blank {max(noise):.3f}, plus 0.005 margin)")
print(f"   for comparison, 0929 had to assume 0.055 from the paper texture under the tray.")

strongest 30-100 kDa peak in each UNLOADED lane: lane 1 0.014, lane 2 0.011, lane 10 0.010, lane 11 0.015, lane 12 0.012
-> noise floor 0.02  (worst blank 0.015, plus 0.005 margin)
   for comparison, 0929 had to assume 0.055 from the paper texture under the tray.


In [8]:
rows = []
for i, nm in enumerate(LANES):
    f = frac(i); col = profile(i)
    rec = dict(lane=i+1, sample=nm, total=round(float(col[zone].sum()), 1))
    if nm != "P.L":
        ye = y_of(PREDICTED, f); wd = abs(y_of(PREDICTED*1.08, f) - y_of(PREDICTED*0.92, f))/2
        a, b = int(ye-wd), int(ye+wd)
        flank = np.r_[col[max(0, a-3*int(wd)):a], col[b:b+3*int(wd)]]
        p, q = strongest(col, f)
        rec |= dict(predicted_kDa=PREDICTED, y_predicted=int(ye),
                    contrast=round(float(col[a:b].max() - np.median(flank)), 4),
                    band_y=int(p), band_kDa_cubic=round(mw_cubic(p, f), 1),
                    band_kDa_piecewise=round(mw_piecewise(p, f), 1),
                    band_prominence=round(float(q), 4),
                    band_volume=round(volume(i, int(p)), 2), detected=bool(q >= FLOOR))
    rows.append(rec)

table = pd.DataFrame(rows)
table["total_vs_ladder"] = (table["total"]/table.loc[table["sample"].eq("P.L"), "total"].mean()).round(2)

gaps = [profile(i, width=.05, shift=.5)[zone].sum() for i in range(NL-1)]
print(f"blank-gel floor for `total`, from the {len(gaps)} gaps between lanes: "
      f"{np.min(gaps):.0f}-{np.max(gaps):.0f}\n")
print(table.to_string(index=False))

blank-gel floor for `total`, from the 11 gaps between lanes: 8-35

 lane  sample  total  predicted_kDa  y_predicted  contrast  band_y  band_kDa_cubic  band_kDa_piecewise  band_prominence  band_volume detected  total_vs_ladder
    1 (empty)   22.4          42.27       1503.0    0.0062   964.0            75.9                77.7           0.0145        -0.10    False             0.48
    2 (empty)   14.3          42.27       1498.0    0.0037  1626.0            38.5                37.6           0.0115         0.08    False             0.30
    3     P.L   48.7            NaN          NaN       NaN     NaN             NaN                 NaN              NaN          NaN      NaN             1.04
    4     raw   24.3          42.27       1490.0    0.0266  1513.0            43.6                40.2           0.0404         1.09     True             0.52
    5     FT1   21.3          42.27       1486.0    0.0241  1500.0            44.0                40.7           0.0357         1.01     T

### Reading those numbers

**The elutions are empty.** E1 and E2 peak at prominence 0.011, below the 0.020 floor this gel sets
on its own blank lanes. Their totals, 13.4 and 12.9, are *inside* the unloaded range (13.9–22.4) —
E2 is fractionally below the faintest never-loaded lane. There is no evidence that anything at all
was loaded into those two wells, let alone purified protein.

**The flow-through is the load.** The 40–44 kDa band is the strongest feature in both:

| | `raw` (lane 4) | `FT1` (lane 5) |
|---|---|---|
| band, rung-to-rung | 40.2 kDa | 40.7 kDa |
| band, cubic | 43.6 kDa | 44.0 kDa |
| prominence | 0.040 | 0.036 |
| band volume | 1.09 | 1.01 |
| lane total | 24.3 | 21.3 |
| contrast at 42.27 kDa | 0.027 | 0.024 |

Every number agrees within 10 %, which is inside this gel's measurement scatter. **A column that
retained its target would deplete the flow-through.** This one did not, so either the target was not
in the load, or it did not bind.

A second, weaker band sits at **48–50 kDa** in both lanes (prominence 0.025 and 0.023). 0929 saw an
unexplained species at 48–50 kDa too, and 0915 at ~50–52 kDa. Same behaviour here: present in load
and flow-through alike, not retained.

**The wash is nearly clean.** W3 peaks at prominence 0.019, a hair under the floor, at 42–46 kDa.
A third wash should be close to empty, and it is — which also means the target was not being washed
off late.

**One caveat on the floor.** The left edge of the gel is lit less evenly than the rest — visible as
coarse speckle over lanes 1–2 in the figures — so those two blanks are noisier than the others. It
does not matter here: the floor is set by lane 11 (0.015), which sits in the smooth region, and the
five sample lanes are all in the smooth region too.

**Scale check.** The load lane carries half the ladder lane's total OD. Against that, the 0929 CDS12
induced band had prominence 0.106 in a lane of total 102. Nothing on this gel is within an order of
magnitude of that. Whatever is being purified here, there is very little of it.

## 7. Annotated figures

Four files, JPEG q92 at dpi 120: {`uniform`, `flattened`} × {`gray`, `color`}.

- **`uniform`** — one linear contrast stretch across the whole frame, nothing else. Publication-safe
  under iGEM's image rules; use it on the wiki.
- **`flattened`** — divided by the §2 background so the faint lanes are visible. A non-uniform
  correction, disclosed in the caption. Use it to read the gel, not to publish it.
- **`gray`** — luminance only. Cleanest for band position.
- **`color`** — RGB kept, not white-balanced (the gel background is stained blue; balancing on it
  turns the surround yellow). The ladder's orange 70 kDa and green 10 kDa rungs show as themselves.

The **red box** spans the five sample lanes and its height is the ±9 % calibration window around
42.27 kDa. It follows the ladder tilt. **Black arrowheads** mark the strongest 30–100 kDa band of
any lane that clears the measured noise floor, labelled `rung-to-rung – cubic`. On this gel only
`raw` and `FT1` qualify — and they carry the same band, which is the result.

In [9]:
REF  = {70: "#E8590C", 10: "#2F9E44"}
ORANGE, RED, INK, GREY = "#C2410C", "#DC2626", "#111111", "#6B7280"
SAT, TOL = 1.6, 1.09          # saturation multiplier (flattened colour only); +/-9% box height

BLOCK  = dict(name="CDS12_ANC_Node281   (e281)", mw=PREDICTED, col="#1D4ED8",
              lanes=(3, 7), tag=(4, 6))
GROUPS = [("elution", 6, 7)]          # the only group spanning more than one lane
LEGEND = ("raw = load   ·   FT1 = flow-through   ·   W3 = wash 3   ·   E1, E2 = elution fractions "
          "(these are column fractions, not the ancestral node e1)")

def runs(idx):
    "Contiguous runs in a sorted index list, as (first, last) pairs."
    out = [[idx[0], idx[0]]]
    for i in idx[1:]:
        if i == out[-1][1] + 1: out[-1][1] = i
        else:                   out.append([i, i])
    return out

def gelimg(corr, palette):
    rgb = crop()
    if palette == "gray":
        lum = rgb.mean(2)
        img = lum/background(lum) if corr == "flattened" else lum
        lo, hi = np.percentile(img, [0.3, 99.7] if corr == "flattened" else [0.05, 99.9])
        return np.clip((img - lo)/(hi - lo), 0, 1)
    if corr == "uniform":
        lo, hi = np.percentile(rgb, [1, 99.5])
        return np.clip((rgb - lo)/(hi - lo), 0, 1)
    out = np.dstack([rgb[..., k]/background(rgb[..., k]) for k in range(3)])
    lo, hi = np.percentile(out, [0.5, 99.8])
    out = np.clip((out - lo)/(hi - lo), 0, 1)
    m = out.mean(2, keepdims=True)
    return np.clip(m + (out - m)*SAT, 0, 1)

In [10]:
def draw(corr, palette):
    v = gelimg(corr, palette)
    lx = np.array([lane_x(i, 0) for i in range(NL)]); step = pitch(0)
    HEAD, SIDE = 0.36*H, 0.115*W
    gray = palette == "gray"

    fig = plt.figure(figsize=(16, 16*(H + HEAD)/(W + 2*SIDE)))
    ax  = fig.add_axes([0, 0.042, 1, 0.878])
    ax.imshow(v, cmap="gray" if gray else None, vmin=0 if gray else None,
              vmax=1 if gray else None, aspect="equal", zorder=1)

    # ladder scales, drawn beside each ladder lane and pointing inwards
    for idx, sgn in ((LADDER_IDX[0], -1), (LADDER_IDX[1], 1)):
        ys = rungs(frac(idx))
        for m, y in zip(LADDER_MW, ys):
            edge = lane_x(idx, y) + sgn*pitch(y)*0.46
            t0, t1 = edge + sgn*0.010*W, edge + sgn*0.046*W
            col = REF.get(int(m), ORANGE); big = int(m) in REF
            ax.plot([t0, t1], [y, y], color=col, lw=3.4 if big else 2.4,
                    solid_capstyle="butt", clip_on=False, zorder=6)
            ax.text(edge + sgn*0.066*W, y, f"{m:.0f}", color=col, fontsize=17 if big else 15,
                    fontweight="bold", ha="center", va="center", clip_on=False, zorder=7)
        ax.text(lane_x(idx, ys[0]) + sgn*0.066*W, ys[0] - 0.055*H, "kDa", color=ORANGE, fontsize=14,
                style="italic", fontweight="bold", ha="center", va="center", clip_on=False, zorder=7)

    ax.add_patch(plt.Rectangle((-SIDE, -HEAD), W + 2*SIDE, HEAD, fc="white", ec="none", zorder=2))
    Y = dict(name=-.286, pred=-.238, brk=-.202, cond=-.146, cbrk=-.119, num=-.038)
    i0, i1 = BLOCK["lanes"]; a, z = lx[i0] - step*.44, lx[i1] + step*.44
    ax.plot([a, a, z, z], [Y["brk"]*H + .020*H, Y["brk"]*H, Y["brk"]*H, Y["brk"]*H + .020*H],
            color=BLOCK["col"], lw=3.0, zorder=6)
    ax.text((a + z)/2, Y["name"]*H, BLOCK["name"], color=BLOCK["col"], fontsize=23,
            fontweight="bold", ha="center", va="center", zorder=6)
    ax.text((a + z)/2, Y["pred"]*H, f"predicted {BLOCK['mw']:.2f} kDa", color=RED, fontsize=16,
            fontweight="bold", ha="center", va="center", zorder=6)
    for cond, j0, j1 in GROUPS:
        a2, z2 = lx[j0] - step*.40, lx[j1] + step*.40
        ax.plot([a2, a2, z2, z2], [Y["cbrk"]*H + .016*H, Y["cbrk"]*H, Y["cbrk"]*H,
                                   Y["cbrk"]*H + .016*H], color=INK, lw=2.0, zorder=6)
        ax.text((a2 + z2)/2, Y["cond"]*H, cond, color=INK, fontsize=16, fontweight="bold",
                ha="center", va="center", zorder=6)
    for i, x in enumerate(lx):
        nm = LANES[i]; blank = nm == "(empty)"
        if not blank:
            ax.text(x, Y["num"]*H - .050*H, nm, color="#374151", fontsize=15.5,
                    fontweight="bold", ha="center", va="center", zorder=6)
        ax.add_patch(plt.Circle((x, Y["num"]*H), step*.185,
                                fc="#9CA3AF" if blank else ("#6B7280" if nm == "P.L" else "#111111"),
                                ec="none", zorder=6))
        ax.text(x, Y["num"]*H, str(i + 1), color="white", fontsize=15, fontweight="bold",
                ha="center", va="center", zorder=7)
    for j0, j1 in runs(EMPTY):            # one label per contiguous run of unloaded wells
        ax.text((lx[j0] + lx[j1])/2, Y["num"]*H - .050*H, "not loaded", color=GREY, fontsize=12.5,
                style="italic", ha="center", va="center", zorder=6)
    ax.text(-SIDE*.94, Y["num"]*H, "lane", color=INK, fontsize=13, fontweight="bold",
            ha="left", va="center", zorder=6)

    fs  = np.linspace(i0 - .46, i1 + .46, 40)
    top = np.array([y_of(BLOCK["mw"]*TOL, frac(i)) for i in fs])
    bot = np.array([y_of(BLOCK["mw"]/TOL, frac(i)) for i in fs])
    xt  = np.array([lane_x(i, y) for i, y in zip(fs, top)])
    xb  = np.array([lane_x(i, y) for i, y in zip(fs, bot)])
    ring = np.c_[np.r_[xt, xb[::-1]], np.r_[top, bot[::-1]]]
    ax.add_patch(Polygon(ring, closed=True, fc=RED, alpha=0.07, ec="none", zorder=4))
    ax.add_patch(Polygon(ring, closed=True, fc="none", ec=RED, lw=3.2, joinstyle="miter", zorder=5))
    t0, t1 = BLOCK["tag"]; xm = (lane_x(t0, bot.mean()) + lane_x(t1, bot.mean()))/2
    ax.text(xm, bot.max() + 0.012*H, f"{BLOCK['mw']:.2f} kDa ±9%", color=RED, fontsize=13,
            fontweight="bold", ha="center", va="top", zorder=6,
            bbox=dict(fc="white", ec=RED, lw=1.5, boxstyle="round,pad=0.26"))

    for _, r in table.iterrows():
        if r["sample"] in ("P.L", "(empty)") or not r["detected"]: continue
        i = int(r["lane"]) - 1; y = r["band_y"]; x = lane_x(i, y) + pitch(y)*.47
        s = step*.13
        ax.add_patch(Polygon([[x, y], [x + s, y - s*.62], [x + s, y + s*.62]], closed=True,
                             fc=INK, ec="white", lw=1.0, zorder=8))
        ax.text(x + s*.2, y - s*.95, f"{r['band_kDa_piecewise']:.0f}–{r['band_kDa_cubic']:.0f}",
                color=INK, fontsize=11.5, fontweight="bold", ha="center", va="bottom", zorder=8,
                bbox=dict(fc="white", ec="none", alpha=.78, boxstyle="round,pad=0.12"))

    ax.set_xlim(-SIDE, W + SIDE); ax.set_ylim(H, -HEAD); ax.axis("off")
    fig.text(.5, .985, "2026-10-10 SDS-PAGE, Coomassie R-250  ·  e281 Ni-NTA fractions  ·  ladder: "
             "PageRuler Prestained, 170–10 kDa (70 kDa orange, 10 kDa green)",
             fontsize=15, fontweight="bold", ha="center", va="top")
    fig.text(.5, .955, LEGEND + "\nred box = where the construct's band should fall, widened by the "
             f"±9 % ladder calibration error   ·   ◀ = strongest band at 30–100 kDa clearing the "
             f"measured noise floor ({FLOOR}), apparent kDa (rung-to-rung – cubic fit)",
             fontsize=12, color=GREY, ha="center", va="top", linespacing=1.6)
    foot = ("background-flattened for visibility: each channel divided by a 400 px, 80th-percentile "
            "estimate of the clear-gel background" if corr == "flattened" else
            "uniform linear contrast stretch only — no background correction")
    if palette == "color":
        foot += (f"; saturation lifted {SAT:.1f}x globally" if corr == "flattened"
                 else "; the same stretch on all three channels, colours as photographed")
    foot += ". No local or per-lane edits."
    fig.text(.5, .006, foot, fontsize=11.5, color=GREY, ha="center", va="bottom", style="italic")

    p = OUT/f"nta-fractions_annotated_{corr}_{palette}.jpg"
    fig.savefig(p, dpi=120, bbox_inches="tight", facecolor="white", pil_kwargs={"quality": 92})
    plt.close(fig)
    return p

for corr in ("uniform", "flattened"):
    for palette in ("gray", "color"):
        print("wrote", draw(corr, palette).relative_to(ROOT))

wrote code/output/1010_gel_annotation/nta-fractions_annotated_uniform_gray.jpg


wrote code/output/1010_gel_annotation/nta-fractions_annotated_uniform_color.jpg


wrote code/output/1010_gel_annotation/nta-fractions_annotated_flattened_gray.jpg


wrote code/output/1010_gel_annotation/nta-fractions_annotated_flattened_color.jpg


## 8. Save the measurements

In [11]:
p = OUT/"nta-fractions_lane_quant.tsv"; table.to_csv(p, sep="\t", index=False); print(p.relative_to(ROOT))

cal = pd.DataFrame([dict(band=i+1, mw_kDa=m, y_left=yl, y_right=yr)
                    for i, (m, yl, yr) in enumerate(zip(LADDER_MW, CAL["yL"], CAL["yR"]))])
p = OUT/"ladder_calibration.tsv"; cal.to_csv(p, sep="\t", index=False); print(p.relative_to(ROOT))

code/output/1010_gel_annotation/nta-fractions_lane_quant.tsv
code/output/1010_gel_annotation/ladder_calibration.tsv


---
*Log — 2026-10-11: first pass on the 2026-10-10 e281 Ni-NTA fraction gel, same pipeline as 0929
(80th-percentile 400 px background, optical density, rung-to-rung beside cubic calibration, ladder
confirmed again as PageRuler Prestained by the red 4th and green 10th rung on both ladder lanes).
Three changes, all forced by this gel: ladders are lanes 3 and 9 rather than the outermost lanes, so
the lane grid is built outwards from them and checked against the gel's own edges; five wells were
never loaded, so the noise floor is measured (0.020) instead of assumed as in 0929 (0.055); one
construct, one block. Result: the elutions E1 and E2 are indistinguishable from blank gel, and the
load and the flow-through carry the same 40–44 kDa band at the same strength (volume 1.09 vs 1.01),
so nothing was retained by the column. The 48–50 kDa species seen on 0915 and 0929 appears again in
both load and flow-through. The gel cannot say whether the cause is no expression (0929 already found
no e281 band in crude lysate), insolubility, or a failed column. Open items: (1) insoluble/pellet
lane — UNDONE; (2) blank BL21(DE3) ±IPTG control lane — UNDONE, third gel running without it;
(3) W1, W2, E3 not loaded; (4) the label sheet records no date, gel %, run conditions, sample
volumes, or the imidazole concentration in the wash and elution buffers; (5) "raw" is not defined as
whole-cell or cleared lysate; (6) the construct identity is not written on the sheet — e281 comes
from the user. Inputs were copied from `1010_sdsGel/` and renamed; the originals are untouched.*